<a href="https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*


### Research question

**Which pages should a content team review first for a potential content refresh when review time is limited?**

### Decision supported

The project supports the decision of **which pages to prioritise for human review**. Rather than trying to predict whether Google will reward a future content change, the aim is to produce a ranked review queue using observable search and engagement signals.

The output is a page-level score and ranking that helps a content team decide where to spend its limited review time first. The ranking is intended as **decision-support**, with the final refresh decision remaining with a human reviewer.

## 2. Data

*Which release, which tables, date windows, what you excluded and why.
Public-safe.*

### Dataset and release

The analysis uses the **FlyRank ML Internship warehouse dataset**, release
`flyrank_pseudonymized_warehouse_release_v20260703`.

The primary source is the daily content-performance table
`fact_content_daily_performance`. Each observation represents a
`report_date + client_hash_id + content_hash_id` combination.

The warehouse snapshot covers daily observations from **27 January 2025 to
30 June 2026**. For this project, I deliberately restricted the analysis to
**March 2026** so that the ranking, features and evaluation were all defined
within a single fixed decision window.

### Analysis window

The March 2026 slice contains **9,841,378 observations** before the
availability filters used for modelling. The modelling dataset was restricted
to observations with usable search-performance data, producing **331,437
page-level observations**.

The main signals used were:

- Google Search Console impressions
- Google Search Console clicks / CTR
- Google Search Console average position
- GA4 sessions
- GA4 engagement rate

Data-availability indicators were examined during the signal audit because
missingness was not random, but they were not used as model features.

### What was excluded

Several types of information were deliberately excluded.

**Future observations** after March 2026 were excluded from the modelling
decision window. This prevents later performance information from influencing
a March ranking.

**Product decision flags** such as health scores, priority scores, action
types and refresh tiers were excluded because these are workflow or product
labels rather than independently observed inputs to the page-ranking problem.

**Client and content identifiers** were not used as predictive features.
`client_hash_id` was used only to create the grouped train/test split, while
`content_hash_id` was retained only to identify pages in the output.

The project also does not use private search queries, client names, page URLs
or other identifying information. The analysis therefore uses the
pseudonymised, aggregate performance fields provided in the internship
dataset and is designed to remain **public-safe**.

### Scope of the data

This is an observational snapshot of search and engagement performance. It
does not contain a reliable post-refresh outcome showing whether a particular
page actually benefited from being refreshed. Consequently, the analysis
uses the available March signals to construct a review-prioritisation
ranking rather than claiming to estimate the causal effect of refreshing a
page.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Modelling objective and assumptions

The modelling objective was to produce a **ranking of pages for review prioritisation**, rather than to predict a guaranteed outcome from refreshing a page.

The main assumption is that observable search and engagement signals can provide useful information for identifying pages that warrant human investigation. However, these signals can also reflect factors outside the content itself, such as search competition, so a high model score is treated as a prioritisation signal rather than a diagnosis.

### Features

The final model used five features derived from the March 2026 observations:

* `log_impressions`
* `gsc_ctr`
* `gsc_avg_position`
* `log_sessions`
* `log_engagement`

Log transformations were applied to the highly skewed impression, session and engagement measures. Client and content identifiers were excluded from the predictive features.

### Proxy label

Because the dataset does not provide a reliable future outcome measuring whether a content refresh actually improved a page, I constructed a **March 2026 proxy target**.

For each client, pages were ranked by their March values for:

* impressions
* CTR
* average position
* engagement rate

A page was labelled `1` when it had **top-quartile impression visibility** and at least one weaker signal: bottom-quartile CTR, bottom-quartile engagement, or top-quartile average-position value. All other pages were labelled `0`.

This produced **12,064 positive observations and 319,373 negative observations** (a positive rate of approximately 3.6%).

The proxy is deliberately treated as a modelling device rather than ground truth. In particular, it is constructed from the same underlying March signals that are supplied to the model.

### Baseline

Before training the machine-learning model, I created a transparent rule-based baseline. The baseline assigns a score using observable March search-performance signals and produces reason codes describing why a page is prioritised.

This provides a simple human-readable benchmark against which the more flexible model can be compared. The baseline is intended to be **honestly beatable**, rather than to represent an existing production system.

### Validation design

The final model was a **Random Forest classifier** with:

* 200 trees
* maximum depth of 10
* minimum leaf size of 25
* balanced class weights
* fixed random seed of 42

The data was split **by client rather than by individual row**. All pages belonging to a client were kept entirely in either the training or test set. This produced:

* **300,880 training observations from 44 clients**
* **30,557 test observations from 11 clients**

This design avoids having pages from the same client appear in both sets, which could make evaluation artificially easy through client-specific similarity. It is therefore a more conservative test of whether the ranking can transfer to unseen clients.

A row-level random split was also examined as a validation check. It produced a higher ROC-AUC (0.966) than the grouped split (0.880), supporting the decision to use the grouped-by-client result as the more appropriate evaluation.

### Leakage checks

Several checks were performed before interpreting the results:

1. **Future-date check:** no observations after March 2026 were used to construct the March model or target.
2. **Client overlap:** there were zero clients shared between the training and test sets.
3. **Identifier check:** `content_hash_id` was not used as a predictive feature, and `client_hash_id` was used only for grouping.
4. **Product-flag check:** unavailable workflow/product labels such as priority or refresh flags were not used.
5. **Feature check:** the final predictive feature set contained only the intended search and engagement variables.
6. **Missingness check:** GSC/GA4 availability was investigated as a potential source of bias but was not included as a predictive feature.

The most important remaining methodological limitation is **proxy-target overlap**: the target is constructed from the same March signals used as features. Therefore, the model evaluation measures how well the model reproduces this constructed prioritisation pattern; it does **not** demonstrate that the model predicts which pages will benefit from a future content refresh.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Model versus baseline

The Random Forest produced a substantially different ranking from the
transparent rule-based baseline. Both were applied to the same held-out
test set of **30,557 pages from 11 unseen clients**.

| Measure | Result |
|---|---:|
| Test observations | 30,557 |
| Test clients | 11 |
| ROC-AUC | 0.880 |
| Average precision | 0.358 |
| Spearman rank correlation with baseline | 0.428 |
| Top-20 overlap between model and baseline | 0 pages |

The grouped-by-client model achieved a ROC-AUC of **0.880** and average
precision of **0.358** against the March 2026 proxy target. The model and
baseline rankings had a Spearman rank correlation of **0.428**, indicating
that they captured some common ordering while still producing substantially
different rankings. Among the 20 highest-ranked pages from each approach,
there were **no overlapping pages**.

The model's strongest feature was `gsc_avg_position`, which accounted for
approximately **68% of the model's feature importance**, followed by
`log_impressions` at approximately **24%**. The remaining features had much
smaller contributions.

This suggests that the model primarily learned to prioritise pages with
meaningful search visibility but relatively weak search positions, whereas
the simpler baseline produced a different ranking based on its explicit
rules.

### Interpreting the comparison

The comparison demonstrates that the machine-learning approach learns a
different and more flexible ranking from the hand-written baseline. This is
useful as a modelling result, but it should not be interpreted as proof that
the Random Forest is better at selecting pages that will benefit from a
content refresh.

The reason is that the proxy target was constructed from the same March 2026
search and engagement signals supplied to the model. The ROC-AUC and average
precision therefore measure how well the model reproduces that constructed
pattern, rather than how accurately it predicts a future refresh outcome.

For the current evidence, the strongest conclusion is therefore:

> **The model provides a different, directional ranking from the transparent
> baseline on held-out clients, with position and search visibility driving
> most of the learned ranking. However, the available evaluation does not
> establish that either ranking predicts the future benefit of refreshing a
> page.**

## 5. Limitations

*What this work cannot claim.*


The results support a limited set of claims. Several important claims cannot
be made from this analysis.

### No causal claim about content refreshes

The analysis does not show that refreshing a page will improve its search
performance. The dataset does not provide a reliable post-refresh outcome
that identifies whether a page benefited from a refresh. Therefore, a high
model score should not be interpreted as evidence that refreshing that page
will cause its performance to improve.

### The target is a proxy, not ground truth

The model was trained against a March 2026 proxy target constructed from
impressions, CTR, average position and engagement. These same underlying
signals are also used as model features.

Consequently, the reported ROC-AUC and average precision measure how well the
model reproduces the constructed March prioritisation pattern. They do not
measure its ability to predict a future refresh outcome.

### Limited time window

The modelling analysis is restricted to March 2026. Although this avoids
using information from later months, it also means that the results do not
establish that the same relationships will hold across different months,
seasonal periods or future search conditions.

### Unseen-client validation is still limited

The grouped-by-client split provides a more conservative evaluation by
holding out entire clients, but the test set contains only 11 clients.
Performance on these clients should therefore not be treated as evidence of
general performance across the full population of websites.

### Observational data and confounding

Search position, CTR and engagement can be influenced by factors that are not
captured by the model, including search competition, query intent, brand
strength and other characteristics of the page or market.

For example, poor search position does not necessarily indicate that a page
is outdated or that changing its content would improve its ranking.

### Missing and uneven data availability

GSC and GA4 data are not available uniformly across observations. The signal
audit showed that data availability was associated with substantially
different observed performance, meaning that missingness is potentially
informative. The final model does not explicitly model these availability
flags.

### No automatic decision-making

The ranking is designed to support a human review process. It should not be
used to automatically rewrite, publish, delete, redirect or otherwise modify
pages. Human reviewers should consider search intent, content quality,
competition and business context before deciding whether a refresh is
appropriate.

### Overall claim boundary

The strongest claim supported by this work is that the model produces a
**different, directional ranking of March 2026 pages for review
prioritisation**, particularly influenced by search position and visibility.

It does **not** establish that the highest-ranked pages are the pages most
likely to benefit from a content refresh, nor that the model will improve
content outcomes if deployed without further validation.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The model output is converted into a ranked review queue for a content or SEO
team. The queue is intended to answer **which pages should be investigated
first**, not which pages should automatically be refreshed.

### Recommended review order

Pages are ranked by the model score. Each page is then assigned a reason code
based on the observable March 2026 signals that support its prioritisation.

| Reason code | Review signal | Recommended human action |
|---|---|---|
| `high_visibility_low_ctr` | High impressions with relatively low CTR | Review title, meta description and search-result alignment |
| `high_visibility_poor_position` | High impressions with relatively poor average position | Review content relevance, depth and search intent |
| `low_engagement` | Relatively low engagement | Review content usefulness and page experience |
| `mixed_signals` | No single dominant signal | Conduct a broader manual content review |

The resulting queue contains **30,557 held-out pages**. The most highly
ranked pages are predominantly pages with relatively high search visibility
and weaker average search position.

For example, the highest-ranked page in the held-out queue had a model score
of **0.9988**, approximately **25.7K impressions**, a **0.023% CTR**, and an
average position of approximately **29.8**. Its reason code was
`high_visibility_poor_position`, leading to the recommendation:

**Review content relevance, depth and search intent.**

### How the queue should be used

The recommended workflow is:

1. **Start with the highest-ranked pages.**
2. **Use the reason code to understand why the page was prioritised.**
3. **Inspect the page and its search intent manually.**
4. **Consider competition and the wider search context.**
5. **Check business importance before investing review time.**
6. Decide whether to **refresh, investigate further, or leave unchanged**.

The model should therefore act as a **triage mechanism**: it reduces the search
space for a content team, while the final decision remains with a human
reviewer.

### What the ranking does not recommend

A high-ranked page should not automatically be:

- rewritten;
- republished;
- deleted or redirected;
- given a new title or meta description;
- labelled as outdated or low quality; or
- assumed to benefit from a content refresh.

These actions require evidence that is not contained in the model. The ranked
queue should instead be treated as a starting point for targeted human
investigation.

### Queue top 20

| Metric | Count |
|---|---:|
| Queue rows | 30,557 |
| Low engagement | 25,553 |
| High visibility + low CTR | 2,175 |
| Mixed signals | 1,688 |
| High visibility + poor position | 1,141 |



| Rank | ML score | Reason code | Recommended action | Impressions | CTR | Avg. position | GA4 sessions | Engagement rate |
|---:|---:|---|---|---:|---:|---:|---:|---:|
| 1 | 0.998408 | high_visibility_poor_position | Review content relevance, depth and search intent | 25,711 | 0.000233 | 29.845 | 28 | 0.035714 |
| 2 | 0.997157 | high_visibility_poor_position | Review content relevance, depth and search intent | 11,420 | 0.000263 | 38.534 | 3 | 0.000000 |
| 3 | 0.997137 | high_visibility_poor_position | Review content relevance, depth and search intent | 8,362 | 0.000239 | 33.502 | 3 | 0.000000 |
| 4 | 0.997002 | high_visibility_poor_position | Review content relevance, depth and search intent | 12,665 | 0.000237 | 56.475 | 6 | 0.000000 |
| 5 | 0.996908 | high_visibility_poor_position | Review content relevance, depth and search intent | 45,653 | 0.001336 | 37.470 | 47 | 0.063830 |
| 6 | 0.996881 | high_visibility_poor_position | Review content relevance, depth and search intent | 12,511 | 0.000240 | 50.673 | 3 | 0.000000 |
| 7 | 0.996526 | high_visibility_poor_position | Review content relevance, depth and search intent | 18,398 | 0.000054 | 48.569 | 5 | 0.000000 |
| 8 | 0.996391 | high_visibility_poor_position | Review content relevance, depth and search intent | 9,815 | 0.000408 | 39.228 | 7 | 0.000000 |
| 9 | 0.996366 | high_visibility_poor_position | Review content relevance, depth and search intent | 5,998 | 0.000333 | 37.335 | 3 | 0.000000 |
| 10 | 0.996203 | high_visibility_poor_position | Review content relevance, depth and search intent | 6,327 | 0.000316 | 31.406 | 4 | 0.000000 |
| 11 | 0.996184 | high_visibility_poor_position | Review content relevance, depth and search intent | 11,820 | 0.000338 | 28.136 | 3 | 0.000000 |
| 12 | 0.996172 | high_visibility_poor_position | Review content relevance, depth and search intent | 6,837 | 0.000439 | 39.233 | 4 | 0.000000 |
| 13 | 0.996154 | high_visibility_poor_position | Review content relevance, depth and search intent | 8,876 | 0.001239 | 29.104 | 30 | 0.033333 |
| 14 | 0.995974 | high_visibility_poor_position | Review content relevance, depth and search intent | 8,511 | 0.000705 | 30.386 | 11 | 0.000000 |
| 15 | 0.995928 | high_visibility_poor_position | Review content relevance, depth and search intent | 6,775 | 0.000148 | 75.166 | 1 | 0.000000 |
| 16 | 0.995865 | high_visibility_poor_position | Review content relevance, depth and search intent | 40,212 | 0.000497 | 34.968 | 24 | 0.000000 |
| 17 | 0.995854 | high_visibility_poor_position | Review content relevance, depth and search intent | 18,249 | 0.000822 | 35.840 | 11 | 0.000000 |
| 18 | 0.995802 | high_visibility_poor_position | Review content relevance, depth and search intent | 10,296 | 0.000097 | 39.927 | 2 | 0.000000 |
| 19 | 0.995650 | high_visibility_poor_position | Review content relevance, depth and search intent | 6,382 | 0.000470 | 45.425 | 5 | 0.000000 |
| 20 | 0.995588 | high_visibility_poor_position | Review content relevance, depth and search intent | 14,875 | 0.000336 | 42.442 | 8 | 0.125000 |

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*


The paper embeds a small set of charts and tables selected to communicate the
main modelling result clearly without overloading the reader.

### Tables

**Table 1 — Model versus baseline**

The main results table compares the Random Forest with the transparent
rule-based baseline on the same held-out test set.

| Measure | Result |
|---|---:|
| Test observations | 30,557 |
| Test clients | 11 |
| ROC-AUC | 0.880 |
| Average precision | 0.358 |
| Spearman rank correlation with baseline | 0.428 |
| Top-20 overlap | 0 pages |

**Table 2 — Ranked review queue**

A summary of the final action queue shows how many pages fall into each
reason-code category.

| Reason code | Count |
|---|---:|
| `low_engagement` | 25,553 |
| `high_visibility_low_ctr` | 2,175 |
| `mixed_signals` | 1,688 |
| `high_visibility_poor_position` | 1,141 |
| **Total** | **30,557** |

**Table 3 — Example top-ranked pages**

The paper includes a small sample of the highest-ranked pages to make the
recommendation output concrete. Client and content identifiers are excluded
from the public-facing table.

### Charts

**Figure 1 — Model score distribution**

A histogram of the Random Forest scores on the held-out test set. This shows
how the model separates pages across the ranking rather than presenting only
the highest-ranked examples.

**Figure 2 — Feature importance**

A horizontal bar chart showing the Random Forest feature importances. This
highlights that `gsc_avg_position` and `log_impressions` account for most of
the model's learned importance.

**Figure 3 — Model ranking versus baseline ranking**

A scatter plot comparing the model score with the baseline score on the
held-out test set. The figure illustrates that the two approaches share some
ordering while producing substantially different rankings.

### Public-safety and presentation checks

All charts and tables shown on the deployed page use aggregate or
pseudonymised information only. Client names, page URLs, private search
queries and other identifying information are not included.

Each figure is accompanied by a short takeaway explaining what the reader
should learn from it. The charts are intended to support the central question
of which pages should be reviewed first, rather than to imply that the model
predicts the causal benefit of a content refresh.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
